### Cốt lõi của thuật toán (DFS)

In [2]:
from collections import deque
import heapq

class Node:
    def __init__(self, state, parent=None, action=None, cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost

    def __lt__(self, other):
        return self.cost < other.cost

    def get_path(self):
        path = []
        cur_node = self
        while cur_node:
            path.append((cur_node.state, cur_node.action))
            cur_node = cur_node.parent
        return path[::-1]

class Problem:
    def __init__(self, start_state, target_state, get_moves, apply_move, step_cost=None):
        self.start_state = start_state
        self.target_state = target_state
        self.get_moves = get_moves
        self.apply_move = apply_move
        self.step_cost = step_cost

def is_goal(problem, cur_state):
    if callable(problem.target_state):
        return problem.target_state(cur_state)
    return cur_state == problem.target_state

def DFS(problem: Problem):
    initial_node = Node(problem.start_state)

    frontier = [initial_node]
    explored = {initial_node.state}

    while frontier:
        cur_node = frontier.pop()

        if is_goal(problem, cur_node.state):
            return cur_node.get_path()

        for action in problem.get_moves(cur_node.state):
            next_state = problem.apply_move(cur_node.state, action)

            if next_state not in explored:
                explored.add(next_state)
                child = Node(state=next_state, parent=cur_node, action=action)
                frontier.append(child)

    return None

### Bài toán 8-Puzzle

In [3]:
def find_blank(cur_board):
    for row_idx in range(3):
        for col_idx in range(3):
            if cur_board[row_idx][col_idx] == 0:
                return row_idx, col_idx

def actions_8puzzle(cur_board):
    blank_r, blank_c = find_blank(cur_board)
    valid_moves = []

    if blank_r > 0: valid_moves.append("Lên")
    if blank_r < 2: valid_moves.append("Xuống")
    if blank_c > 0: valid_moves.append("Trái")
    if blank_c < 2: valid_moves.append("Phải")

    return valid_moves

def result_8puzzle(cur_board, move):
    blank_r, blank_c = find_blank(cur_board)

    new_r, new_c = blank_r, blank_c
    if move == "Lên":
        new_r -= 1
    elif move == "Xuống":
        new_r += 1
    elif move == "Trái":
        new_c -= 1
    elif move == "Phải":
        new_c += 1

    grid = [list(cur_board[0]), list(cur_board[1]), list(cur_board[2])]
    grid[blank_r][blank_c] = grid[new_r][new_c]
    grid[new_r][new_c] = 0

    return (tuple(grid[0]), tuple(grid[1]), tuple(grid[2]))

def print_state(board):
    for r in range(3):
        print(f"[{board[r][0]} {board[r][1]} {board[r][2]}]")
    print()

if __name__ == "__main__":
    start_board = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
    target_board = ((1, 2, 3), (4, 5, 6), (7, 8, 0))
    puzzle = Problem(start_board, target_board, actions_8puzzle, result_8puzzle)

    steps = DFS(puzzle)
    if steps is None:
        print("Không tìm thấy lời giải")
    else:
        total_moves = len(steps) - 1
        print(f"SOLUTION (Tổng số bước DFS tìm được: {total_moves})")
        if len(steps) <= 10:
            show_indices = list(range(len(steps)))
        else:
            show_indices = [0, 1, 2, -1, len(steps) - 2, len(steps) - 1]

        for step_idx in show_indices:
            if step_idx == -1:
                print(f"... (ẩn bớt {len(steps) - 5} bước ở giữa) ...\n")
                continue
            if step_idx == len(steps) - 1:
                print(f"Bước {step_idx} (GOAL):")
            else:
                print(f"Bước {step_idx}:")
            print_state(steps[step_idx][0])

SOLUTION (Tổng số bước DFS tìm được: 49284)
Bước 0:
[1 2 3]
[4 0 6]
[7 5 8]

Bước 1:
[1 2 3]
[4 6 0]
[7 5 8]

Bước 2:
[1 2 3]
[4 6 8]
[7 5 0]

... (ẩn bớt 49280 bước ở giữa) ...

Bước 49283:
[1 2 3]
[4 5 0]
[7 8 6]

Bước 49284 (GOAL):
[1 2 3]
[4 5 6]
[7 8 0]



### Bài toán Vacuum Robot

In [2]:
def actions_vacuum(cur_state):
    return ["Hút", "Trái", "Phải"]

def result_vacuum(cur_state, move):
    pos, room_a, room_b = cur_state

    if move == "Trái":
        return ("A", room_a, room_b)
    elif move == "Phải":
        return ("B", room_a, room_b)
    elif move == "Hút":
        if pos == "A":
            return (pos, "Sạch", room_b)
        else:
            return (pos, room_a, "Sạch")

def goal_vacuum(cur_state):
    return cur_state[1] == "Sạch" and cur_state[2] == "Sạch"

def print_vacuum_state(cur_state):
    pos, room_a, room_b = cur_state
    print(f"Robot ở phòng {pos} | Phòng A: {room_a} | Phòng B: {room_b}")

if __name__ == "__main__":
    start_vacuum = ("A", "Bẩn", "Bẩn")
    vacuum_puzzle = Problem(start_vacuum, goal_vacuum, actions_vacuum, result_vacuum)

    steps = DFS(vacuum_puzzle)
    if steps is None:
        print("Không tìm thấy lời giải")
    else:
        print("SOLUTION")
        for step_idx in range(len(steps)):
            if step_idx == len(steps) - 1:
                print(f"Bước {step_idx} (GOAL):")
            else:
                print(f"Bước {step_idx}:")
            print_vacuum_state(steps[step_idx][0])

SOLUTION
Bước 0:
Robot ở phòng A | Phòng A: Bẩn | Phòng B: Bẩn
Bước 1:
Robot ở phòng B | Phòng A: Bẩn | Phòng B: Bẩn
Bước 2:
Robot ở phòng B | Phòng A: Bẩn | Phòng B: Sạch
Bước 3:
Robot ở phòng A | Phòng A: Bẩn | Phòng B: Sạch
Bước 4 (GOAL):
Robot ở phòng A | Phòng A: Sạch | Phòng B: Sạch
